# DiFuMo MLP Colab Controls

Trains only the DiFuMo-512 coefficient MLP controls. Run this in its own Colab session, then run `experiments/difumo_gat_colab.ipynb` for graph models and `experiments/difumo_compare_colab.ipynb` for saved-artifact comparisons.

In [ ]:
import os, platform, sys
print('Python:', sys.version)
try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('Torch check failed:', exc)
print('Platform:', platform.platform())

In [ ]:
!pip -q install nilearn nibabel scipy huggingface-hub safetensors adapters transformers pyarrow matplotlib pandas scikit-learn tqdm umap-learn torch-geometric

In [ ]:
REPO_URL = 'https://github.com/neurovlm/neurovlm.git'
REPO_BRANCH = 'neurovlm_gnn'
REPO_DIR = '/content/neurovlm_gnn'
if not os.path.exists(REPO_DIR):
    !git clone --branch $REPO_BRANCH --single-branch $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR fetch origin $REPO_BRANCH
    !git -C $REPO_DIR checkout $REPO_BRANCH
    !git -C $REPO_DIR pull --ff-only origin $REPO_BRANCH
os.chdir(REPO_DIR)
!pip -q install -e '.[viz,notebook,metrics]'
print('Working directory:', os.getcwd())

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configure MLP Runs

This notebook trains coefficient-only MLP controls. It uses `runs_difumo_mlp`, `data_difumo_mlp`, and `checkpoints_difumo_mlp` so it can live next to the DiFuMo GAT notebook without sharing output folders. Semantic evaluation reads the shared Drive resource folder at `/content/drive/MyDrive/neurovlm_evaluation_resources`.


In [ ]:
from pathlib import Path
import os, time

MODEL_NAME = 'difumo_mlp'
DRIVE_ROOT = '/content/drive/MyDrive/neurovlm_difumo_mlp'
DRIVE_DATA_DIR = f'{DRIVE_ROOT}/data_{MODEL_NAME}'
RUN_ROOT = f'{DRIVE_ROOT}/runs_{MODEL_NAME}'
RAW_COEFF_CACHE = f'{DRIVE_DATA_DIR}/difumo512_pubmed_flatmap_coeffs.npz'
ALE_COEFF_CACHE = f'{DRIVE_DATA_DIR}/difumo512_pubmed_ale_fwhm9_coeffs.npz'
COMPARISON_FILE = f'{RUN_ROOT}/difumo_mlp_comparison.csv'
EVAL_RESOURCE_DIR = '/content/drive/MyDrive/neurovlm_evaluation_resources'
# To resume a previous interrupted batch, set this to that folder name.
RESUME_BATCH_NAME = None
RUN_BATCH_NAME = RESUME_BATCH_NAME or time.strftime('%Y%m%d_%H%M%S_mlp')

os.makedirs(RUN_ROOT, exist_ok=True)
os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
os.makedirs(EVAL_RESOURCE_DIR, exist_ok=True)

MLP_BATCH_SIZE = 1024
SHOW_COMMAND = False
FORCE_RERUN = False
STOP_ON_FAILURE = True

BASE = dict(
    model='mlp',
    epochs=80,
    batch_size=MLP_BATCH_SIZE,
    mlp_hidden_dim=1024,
    lr_gat=3e-4,
    lr_proj=3e-5,
    temperature=0.07,
    early_stopping_patience=12,
    early_stopping_monitor='train_loss',
    early_stopping_min_delta=1e-4,
    val_interval=1,
    coeff_source='flatmap',
    coeff_cache_file=RAW_COEFF_CACHE,
    amp=True,
    umap=True,
)

CONFIGS = [
    dict(BASE, name='difumo_mlp_raw_coeff512'),
    dict(BASE, name='difumo_mlp_ale_fwhm9_coeff512', coeff_source='ale_coordinates', coeff_cache_file=ALE_COEFF_CACHE, ale_kernel_fwhm_mm=9),
]

RUN_CONFIGS = CONFIGS
print('Drive root:', DRIVE_ROOT)
print('Drive data directory:', DRIVE_DATA_DIR)
print('Run root:', RUN_ROOT)
print('Comparison file:', COMPARISON_FILE)
print('Evaluation resource directory:', EVAL_RESOURCE_DIR)
len(RUN_CONFIGS), [c['name'] for c in RUN_CONFIGS]


In [ ]:
import json, os, shlex, subprocess, sys, time

def cli_key(key):
    return '--' + key.replace('_', '-')

def run_config(cfg):
    name = cfg['name']
    run_dir = f'{RUN_ROOT}/{RUN_BATCH_NAME}/{name}'
    checkpoint_dir = f'{run_dir}/checkpoints_{MODEL_NAME}'
    cmd = [
        sys.executable, 'experiments/train_difumo_gat_colab.py',
        '--run-dir', run_dir,
        '--checkpoint-dir', checkpoint_dir,
        '--comparison-file', COMPARISON_FILE,
        '--semantic-eval',
        '--eval-resource-dir', EVAL_RESOURCE_DIR,
    ]
    for key, value in cfg.items():
        if key == 'name' or value is None:
            continue
        if isinstance(value, bool):
            if value:
                cmd.append(cli_key(key))
            continue
        cmd.extend([cli_key(key), str(value)])
    os.makedirs(run_dir, exist_ok=True)
    log_path = f'{run_dir}/subprocess.log'
    done_path = f'{run_dir}/eval_results.json'
    ckpt_path = f'{checkpoint_dir}/best_difumo_gat.pt'
    print(f'\n=== {name} ===')
    print('Run dir:', run_dir)
    print('Log file:', log_path)
    if not FORCE_RERUN and os.path.exists(done_path) and os.path.exists(ckpt_path):
        print('Already completed; skipping. Set FORCE_RERUN=True to train again.')
        return {'name': name, 'status': 'skipped', 'run_dir': run_dir}
    if SHOW_COMMAND:
        print('$', ' '.join(shlex.quote(x) for x in cmd))
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    started = time.time()
    tail = []
    with open(log_path, 'w') as log:
        proc = subprocess.Popen(
            [cmd[0], '-u', *cmd[1:]],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
            tail.append(line.rstrip())
            tail = tail[-80:]
        return_code = proc.wait()
    elapsed_min = (time.time() - started) / 60
    if return_code != 0:
        print(f'\n{name} failed after {elapsed_min:.1f} min. Last log lines:')
        print('\n'.join(tail[-60:]))
        result = {'name': name, 'status': 'failed', 'run_dir': run_dir, 'log_path': log_path, 'return_code': return_code}
        if STOP_ON_FAILURE:
            raise RuntimeError(f'{name} failed with exit code {return_code}. Full log: {log_path}')
        return result
    print(f'{name} finished in {elapsed_min:.1f} min')
    manifest = {
        'name': name,
        'drive_run_dir': run_dir,
        'best_checkpoint': f'{checkpoint_dir}/best_difumo_gat.pt',
        'last_checkpoint': f'{checkpoint_dir}/last_difumo_gat.pt',
        'eval_results': f'{run_dir}/eval_results.json',
        'training_history': f'{run_dir}/training_history.json',
        'config': f'{run_dir}/config.json',
        'artifact_manifest': f'{run_dir}/artifacts_manifest.json',
        'comparison_file': COMPARISON_FILE,
        'main_comparison_summary_row': f'{run_dir}/main_comparison_summary_row.csv',
        'semantic_evaluation_manifest': f'{run_dir}/semantic_evaluation_manifest.json',
        'coefficient_cache': cfg.get('coeff_cache_file'),
        'coefficient_source': cfg.get('coeff_source'),
    }
    with open(f'{run_dir}/colab_drive_manifest.json', 'w') as f:
        json.dump(manifest, f, indent=2)
    print('Saved Drive artifacts under:', run_dir)
    return {'name': name, 'status': 'completed', 'run_dir': run_dir}

run_results = []
for cfg in RUN_CONFIGS:
    run_results.append(run_config(cfg))
run_results
